**Eksperyment 1: Wpływ funkcji aktywacji na aproksymację funkcji w zadaniu regresji**
**Testowane Layery:**
- 0 hidden-layers, 16 perceptronów per layer
- 1 hidden-layers, 16 perceptronów per layer
- 2 hidden-layers,, 16 perceptronów per layer, 16-32, 32-32
- 3 hidden-layers, 16 perceptronów per layer
- 4 hidden-layers, 16 perceptronów per layer


**Każdy eksperyment będzie miał różną architektóre sieci, ale stałe parametry to:**
- aktywacja wyjściowa to f(x) = x
- funkcja aktywacji to sigmoid
- funkcja straty to MSE
- optymalizatior to SGD
- learning rate = 0.005
- ilość epok = 500
- batch = 32


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

from mlp_model.activations.activation import Activation
from mlp_model.activations.relu import ReLU
from mlp_model.activations.swish import Swish
from mlp_model.layers.layer import Layer

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from mlp_model.activations.sigmoid import Sigmoid
from mlp_model.initialize_weights.xavier import XavierInitializer
from mlp_model.layers.hidden import Hidden
from mlp_model.layers.output_layer import OutputLayer
from mlp_model.losses.mse_loss import MSELoss
from mlp_model.mlp import MLP
from mlp_model.optimizers.sgd import SGDOptimizer
from trainer.trainer import MLPTrainer

data_kind = "activation"
data_count = 1000 # 100, 500, 1000, 10000

train_filepath = Path(project_root) / "data/regression" / f"data.{data_kind}.train.{data_count}.csv"
df = pd.read_csv(train_filepath)
x = df["x"].to_numpy().reshape(-1, 1)  # shape: (1000, 1)
y = df["y"].to_numpy().reshape(-1, 1)  # shape: (1000, 1)

In [2]:
def test_layers(l: list[Layer]):
    mlp = MLP(layers=l)
    mlp.initialize_weights(XavierInitializer())
    sgd = SGDOptimizer(mlp, 0.005)
    trainer = MLPTrainer(mlp, sgd, MSELoss(), 500, 32)
    trainer.cross_validate(x, y, 5)

In [3]:
layers = {
    "0": [OutputLayer(1,1)],
    "1": [Hidden(1,16, Sigmoid()), OutputLayer(16,1)],
    "2-8-16": [Hidden(1,8, Sigmoid()), Hidden(8,16, Sigmoid()), OutputLayer(16,1)],
    "2-16-16": [Hidden(1,16, Sigmoid()), Hidden(16,16, Sigmoid()), OutputLayer(16,1)],
    "2-16-32": [Hidden(1,16, Sigmoid()), Hidden(16,32, Sigmoid()), OutputLayer(32,1)],
    "2-32-32": [Hidden(1,32, Sigmoid()), Hidden(32,32, Sigmoid()), OutputLayer(32,1)],
    "3": [Hidden(1,16, Sigmoid()), Hidden(16,16, Sigmoid()), Hidden(16,16, Sigmoid()), OutputLayer(16,1)],
    "4": [Hidden(1,16, Sigmoid()), Hidden(16,16, Sigmoid()), Hidden(16,16, Sigmoid()), Hidden(16,16, Sigmoid()), OutputLayer(16,1)],
}

In [5]:
for k, v in layers.items():
    print(f"--------| {k} |--------")
    test_layers(v)

--------0--------
Fold 1: loss = 669.397730 | metrics {'r2': 0.9123495774268723, 'mae': 23.027827782966504, 'mse': 669.3977299828808}
Fold 2: loss = 694.711547 | metrics {'r2': 0.9098668960388854, 'mae': 23.67614877830606, 'mse': 694.711547202757}
Fold 3: loss = 722.587680 | metrics {'r2': 0.903083225731519, 'mae': 24.10972760841493, 'mse': 722.587679990967}
Fold 4: loss = 755.480307 | metrics {'r2': 0.9095986087038759, 'mae': 24.761775010752963, 'mse': 755.4803066102514}
Fold 5: loss = 705.696189 | metrics {'r2': 0.9156077086920926, 'mae': 23.743784422244786, 'mse': 705.6961886567168}
---------------------------------------------------------------
Cross validation with 5 folds:
Mean loss: 709.574690 (std: 28.729105)
---------------------------------------------------------------
--------1--------
Fold 1: loss = 0.009394 | metrics {'r2': 0.9999987699992888, 'mae': 0.070140450381657, 'mse': 0.009393676148346934}
Fold 2: loss = 0.009216 | metrics {'r2': 0.9999988042947536, 'mae': 0.06640